# Experiment 1 — Word-context representation

In [1]:
import pandas as pd
import re
file_path = '../lab2/c4-train.00000-of-01024-30K.json'
df = pd.read_json(file_path, lines=True)

df_sampled = df.sample(n=10000, random_state=42)

def tokenize(text):
  return re.findall(r'\b\w+\b', str(text).lower())

corpus = df_sampled['text'].apply(tokenize).tolist()
corpus = [doc for doc in corpus if len(doc) > 0]

In [2]:
from cooccurrence import MyCooccurrence

windows = [1, 2, 5]
for window in windows:
  print(f"\n{'='*40}")
  print(f"WINDOW SIZE = {window}")
  print(f"{'='*40}")

  model = MyCooccurrence()
  model.build_vocabulary(corpus, min_count=5)
  model.build_cooccurrence_matrix(corpus, window_size=window)
  matrix_shape = model.matrix.shape
  nnz = model.matrix.nnz
  total_elements = matrix_shape[0] * matrix_shape[1]
  sparsity = 100 - (nnz / total_elements * 100)
  print(f"Vocabulary Size:   {model.vocab_size} words")
  print(f"Matrix Size:       {matrix_shape[0]} x {matrix_shape[1]} cells")
  print(f"Non-zero values:   {nnz:,}")
  print(f"Matrix Sparsity:   {sparsity:.4f}%")

  try:
    doctor_vec = model.matrix[model.word2id["doctor"]].toarray().flatten()
    physician_vec = model.matrix[model.word2id["physician"]].toarray().flatten()
    
    sim = model.cosine_similarity(doctor_vec, physician_vec)
    print(f"Similarity (doctor - physician): {sim:.4f}")
  except KeyError:
    print("Similarity test failed: 'doctor' or 'physician' is not in this random 10k sample.")


WINDOW SIZE = 1
Vocabulary Size:   29381 words
Matrix Size:       29381 x 29381 cells
Non-zero values:   2,099,883
Matrix Sparsity:   99.7567%
Similarity (doctor - physician): 0.8355

WINDOW SIZE = 2
Vocabulary Size:   29381 words
Matrix Size:       29381 x 29381 cells
Non-zero values:   4,078,885
Matrix Sparsity:   99.5275%
Similarity (doctor - physician): 0.8487

WINDOW SIZE = 5
Vocabulary Size:   29381 words
Matrix Size:       29381 x 29381 cells
Non-zero values:   8,885,616
Matrix Sparsity:   98.9707%
Similarity (doctor - physician): 0.8940


In [3]:
test_words = ["doctor", "hospital", "car", "computer", "food"]

for word in test_words:
  print(f"Word: '{word}'")
  try:
    results = model.most_similar(word, top_k=5)
    for rank, (match_word, score) in enumerate(results, 1):
      print(f"  {rank}. {match_word} (score: {score:.4f})")
  except KeyError:
    print(f"'{word}' is not in the vocabulary.")

Word: 'doctor'
  1. child (score: 0.9456)
  2. vehicle (score: 0.9248)
  3. plan (score: 0.9242)
  4. or (score: 0.9220)
  5. home (score: 0.9209)
Word: 'hospital'
  1. house (score: 0.9404)
  2. public (score: 0.9403)
  3. community (score: 0.9373)
  4. office (score: 0.9360)
  5. school (score: 0.9347)
Word: 'car'
  1. while (score: 0.9744)
  2. house (score: 0.9720)
  3. company (score: 0.9693)
  4. after (score: 0.9671)
  5. game (score: 0.9667)
Word: 'computer'
  1. business (score: 0.9407)
  2. home (score: 0.9370)
  3. budget (score: 0.9349)
  4. account (score: 0.9333)
  5. journey (score: 0.9327)
Word: 'food'
  1. music (score: 0.9656)
  2. water (score: 0.9654)
  3. local (score: 0.9634)
  4. other (score: 0.9630)
  5. performance (score: 0.9629)


# Experiment 2 — Word2Vec

In [4]:
from gensim.models import Word2Vec
w2v = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=5, epochs=10, seed=42,workers=4)

In [5]:
test_words = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]
for word in test_words:
  print(f"Word: '{word}'")
  try:
    results = w2v.wv.most_similar(word, topn=5)
    for rank, (sim_word, score) in enumerate(results, 1):
      print(f"  {rank}. {sim_word} ({score:.4f})")
  except KeyError:
    print(f"'{word}' not in vocabulary.")

Word: 'doctor'
  1. physician (0.7949)
  2. spouse (0.7228)
  3. child (0.7066)
  4. novak (0.6359)
  5. husband (0.6327)
Word: 'hospital'
  1. clinic (0.7094)
  2. indiana (0.6886)
  3. cincinnati (0.6782)
  4. yale (0.6694)
  5. alabama (0.6691)
Word: 'patient'
  1. patients (0.6474)
  2. treatment (0.6458)
  3. medication (0.6290)
  4. diagnosis (0.6044)
  5. child (0.5931)
Word: 'disease'
  1. diseases (0.8057)
  2. diabetes (0.8054)
  3. syndrome (0.8039)
  4. chronic (0.8030)
  5. infection (0.7974)
Word: 'computer'
  1. desktop (0.7840)
  2. browser (0.7522)
  3. device (0.7431)
  4. laptop (0.7316)
  5. server (0.6991)
Word: 'football'
  1. baseball (0.8323)
  2. basketball (0.8104)
  3. soccer (0.7679)
  4. championship (0.7237)
  5. club (0.7149)
Word: 'banana'
  1. peppers (0.8969)
  2. onions (0.8944)
  3. cabbage (0.8928)
  4. mushrooms (0.8926)
  5. pudding (0.8900)


# Experiment 3 — Context window

In [6]:
import pandas as pd
word_pairs = [
  ("doctor", "physician"),
  ("doctor", "hospital"),
  ("cat", "dog")
]
results_dict = {
  "Word pair": [f"{w1} - {w2}" for w1, w2 in word_pairs]
}
windows_to_test = [2, 5, 10]


In [8]:
for win in windows_to_test:
  print(f"Word2Vec (Window={win})")
  model_exp3 = Word2Vec(
    sentences=corpus,
    vector_size=100,      
    window=win,           
    min_count=5,
    epochs=10,
    sg=1,                 
    seed=42,
    workers=4
  )
  sim_scores = []
  for w1, w2 in word_pairs:
    try:
      score = model_exp3.wv.similarity(w1, w2)
      sim_scores.append(round(score, 4))
    except KeyError:
      sim_scores.append(None)
  results_dict[f"Window {win}"] = sim_scores


Word2Vec (Window=2)
Word2Vec (Window=5)
Word2Vec (Window=10)


In [9]:
df_exp3 = pd.DataFrame(results_dict)
display(df_exp3)

,Word pair,Window 2,Window 5,Window 10
0,doctor - physician,0.7542,0.6950,0.7472
1,doctor - hospital,0.4390,0.4876,0.5377
2,cat - dog,0.5619,0.6868,0.6441


# Experiment 4 — Embedding dimension

In [10]:
import time
import pandas as pd

dimensions = [50, 100, 300]

results_dim = {
  "Dimension": [],
  "Training Time (s)": [],
  "Vector Matrix Size": [],
  "Sim (doctor-physician)": [],
  "Analogy (king-man+woman)": []
}


for dim in dimensions:
  start_time = time.time()
  model_exp4 = Word2Vec(
    sentences=corpus,
    vector_size=dim,     
    window=5,             
    min_count=5,
    epochs=10,
    sg=1,                 
    seed=42,
    workers=4
  )
  train_time = round(time.time() - start_time, 2)
  vocab_size = len(model_exp4.wv.index_to_key)
  matrix_size = vocab_size * dim
  try:
    sim = model_exp4.wv.similarity("doctor", "physician")
    sim_score = round(sim, 4)
  except KeyError:
    sim_score = None
  try:
    analogy_result = model_exp4.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
    analogy_word = analogy_result[0][0]
  except KeyError:
    analogy_word = "N/A"
  results_dim["Dimension"].append(dim)
  results_dim["Training Time (s)"].append(train_time)
  results_dim["Vector Matrix Size"].append(matrix_size)
  results_dim["Sim (doctor-physician)"].append(sim_score)
  results_dim["Analogy (king-man+woman)"].append(analogy_word)

In [11]:
df_exp4 = pd.DataFrame(results_dim)
display(df_exp4)

,Dimension,Training Time (s),Vector Matrix Size,Sim (doctor-physician),Analogy (king-man+woman)
0,50,136.70,1469050,0.7983,queen
1,100,133.61,2938100,0.7595,queen
2,300,196.40,8814300,0.5575,queen


# Application — Semantic Search

In [15]:
import numpy as np
def get_document_embedding(tokens, w2v_model):
  vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv.key_to_index]
  if len(vectors) == 0:
    return np.zeros(w2v_model.vector_size)
  return np.mean(vectors, axis=0)
doc_embeddings = np.array([get_document_embedding(doc, model_exp4) for doc in corpus])
def semantic_search(query_str, w2v_model, doc_matrix, original_corpus, top_k=3):
  query_tokens = re.findall(r'\b\w+\b', query_str.lower())
  query_vec = get_document_embedding(query_tokens, w2v_model)
  query_norm = np.linalg.norm(query_vec)
  
  if query_norm == 0:
    return "Query words are not in the vocabulary"
  dot_products = doc_matrix.dot(query_vec)
  doc_norms = np.linalg.norm(doc_matrix, axis=1)
  similarities = dot_products / (doc_norms * query_norm + 1e-9)
  best_indices = np.argsort(similarities)[::-1][:top_k]
  print(f"SEARCH QUERY: '{query_str}'")
  for rank, idx in enumerate(best_indices, 1):
      score = similarities[idx]
      doc_text = " ".join(original_corpus[idx])
      print(f"\n[Rank {rank} | Score: {score:.4f}]")
      print(f"{doc_text}\n")
      

In [16]:
semantic_search("medical treatment", model_exp4, doc_embeddings, corpus, top_k=5)
semantic_search("delicious food recipe", model_exp4, doc_embeddings, corpus, top_k=5)
semantic_search("implement linear regression", model_exp4, doc_embeddings, corpus, top_k=5)

SEARCH QUERY: 'medical treatment'

[Rank 1 | Score: 0.6350]
a phase ii pilot study to evaluate use of intravenous lidocaine for opioid refractory pain in cancer patients kerala india status of cancer pain relief and palliative care nurse moral distress and cancer pain management an ethnography of oncology nurses in india


[Rank 2 | Score: 0.5933]
not all ultrasound systems are equal created equal truffles vein specialists has philips affinity ultrasound systems the state of the art cutting edge ultrasound system along with award winning experienced vascular staff enables truffles vein specialists to provide the highest level of vein diagnosis and treatment combining the highest level of medical vascular imaging with a patient centered approach to vein treatment truffles has raised the bar when it comes to vein treatment


[Rank 3 | Score: 0.5858]
on january 28 2016 food and drug administration fda has approved merck and co manufactured zepatier a new oral treatment for adult patients 